# NPS-25-003

Template: Getting_started.ipynb from hepdata_lib

From hepdata_lib:
The following instructions and examples should get you started to get your analysis into [HEPData](https://hepdata.net) using `hepdata_lib`. Please also refer to the [documentation](http://hepdata-lib.readthedocs.io/). While you can also run `hepdata_lib` on your local computer, you can use the [binder](https://mybinder.org/) or [SWAN](http://swan.cern.ch/) services in the browser. Mind that SWAN is only available for people with a CERN account.

Also useful reference: https://github.com/jalimena/HepData_EXO-23-016/tree/main
See "main" function in createHepData_all.py

## General setup

To make sure things are working and `hepdata_lib` is available, run the following command:

In [ ]:
import hepdata_lib
import numpy as np
from hepdata_lib import Submission, Table, Variable
from __future__ import print_function
print("hepdata_lib version", hepdata_lib.__version__)

## Adding a table/figure

In HEPData, figures and table will both be `Table` objects. 

The first column is the mass of phi_2, the second of phi_1, and the third is the median upper limit.

Let's create the table/figure. First, we need to give it a name, which is usually just the identifier in the paper, i.e. "Figure _". The table also needs a description, which is usually the caption. You also need to describe the location, i.e. where to find it in the publication:

In [ ]:
def make2DLimitTable(tableName, isBDT, fileName, imageName):

    table = Table(tableName)
    if isBDT:
        table.description = (
            r"The 95% CL upper limits on the products $\sigma B_\mathrm{C}$ and"
            r" $\sigma B_\mathrm{NC}$, obtained using the BDT-based event categorization,"
            r" as a function of the scalar masses $m_{\phi_1}$ and $m_{\phi_2}$."
            r" For the $(m_{\phi_1}, m_{\phi_2})$ mass hypotheses (15, 30), (20, 40),"
            r" and (30, 60) GeV, only the non-cascade limits are shown. For all other mass"
            r" hypotheses, either cascade or non-cascade limits are presented, depending on"
            r" whether the cascade decay is kinematically allowed"
            r" ($m_{\phi_2} \geq 2 m_{\phi_1}$). The numbers displayed in the plot are in pb."
        )
    else:
        table.description = (
            r"The 95% CL upper limits on the products $\sigma B_\mathrm{C}$ and"
            r" $\sigma B_\mathrm{NC}$, obtained using the cut-based event categorization,"
            r" as a function of the scalar masses $m_{\phi_1}$ and $m_{\phi_2}$."
            r" For the $(m_{\phi_1}, m_{\phi_2})$ mass hypotheses (15, 30), (20, 40),"
            r" and (30, 60) GeV, only the non-cascade limits are shown. For all other mass"
            r" hypotheses, either cascade or non-cascade limits are presented, depending on"
            r" whether the cascade decay is kinematically allowed"
            r" ($m_{\phi_2} \geq 2 m_{\phi_1}$). The numbers displayed in the plot are in pb."
        )

    table.location = "Results"
    table.keywords["observables"] = ["SIG"]
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b"
    ]
    #do I need phrases and "particles"?
    data = np.loadtxt(f"NPS25003_inputs/{fileName}", skiprows=0)

    # Column meaning
    y_vals = data[:, 0]   # FIRST column = y bin centers
    x_vals = data[:, 1]   # SECOND column = x bin centers
    z_vals = data[:, 2]   # bin content

    # Build bin edges from centers
    def make_edges(centers):
        centers = np.unique(centers.astype(float))
        edges = np.zeros(len(centers) + 1)
        edges[1:-1] = 0.5 * (centers[1:] + centers[:-1])
        edges[0] = centers[0] - (edges[1] - centers[0])
        edges[-1] = centers[-1] + (centers[-1] - edges[-2])
        return centers, edges

    y_centers, y_edges = make_edges(y_vals)
    x_centers, x_edges = make_edges(x_vals)

    # Independent variables
    phi2_mass = Variable(
        "phi_2 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    phi1_mass = Variable(
        "phi_1 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    # Map center -> edge tuple
    y_edges_map = {y: (y_edges[i], y_edges[i+1]) for i, y in enumerate(y_centers)}
    x_edges_map = {x: (x_edges[i], x_edges[i+1]) for i, x in enumerate(x_centers)}

    # Only include bins that exist in your data
    phi2_mass.values = [y_edges_map[y] for y in y_vals]
    phi1_mass.values = [x_edges_map[x] for x in x_vals]

    # Dependent variable
    median_limit = Variable(
        "Median limit",
        is_independent=False,
        is_binned=False,
        units="pb"
    )

    median_limit.values = [float(v) for y,x,v in data] 
    median_limit.add_qualifier("SQRT(S)", "13", "TeV")

    # Add to table
    table.add_variable(phi1_mass)
    table.add_variable(phi2_mass)
    table.add_variable(median_limit)

    table.add_image(f"NPS25003_inputs/{imageName}")
    table.add_additional_resource("Original data file", f"NPS25003_inputs/{fileName}", copy_file=True)
    print(table.name)
    return table

In [ ]:
def make1DLimitTable_v13(tableName, m1m2_pairs, subfolder, decay, isBDT, isCascade, imageName, description=None):
    """
    Creates a single combined 1D limit Table for one (method × topology) combination.
    
    Files are per-m1:  higgsCombine_a1a2_{decay}_allchannels_allyears_m1_{m1}_limits.txt
    Each file has columns: mh, obs, exp_m2s, exp_m1s, exp_med, exp_p1s, exp_p2s
    where mh is m2. We extract the row matching m2 from the appropriate m1 file.

    Parameters
    ----------
    tableName   : str
    m1m2_pairs  : list of (m1, m2) tuples defining the x-axis points
    subfolder   : str  e.g. "cutbased_root" or "bdtbased_root"
    decay       : str  "4b2t" or "2b2t"
    isBDT       : bool
    isCascade   : bool
    imageName   : str  path to summary plot (relative to NPS25003_inputs/)
    description : str or None  override table description; falls back to isCascade default
    """
    XS_H = 52.38  # ggF H production cross section in pb

    table = Table(tableName)
    table.description = description if description is not None else (
        r"$\mathcal{B}(\mathrm{H} \to \phi_1 \phi_2 \to 2\tau\,4b)$ (%)" if isCascade
        else r"$\mathcal{B}(\mathrm{H} \to \phi_1 \phi_2 \to 2\tau\,2b)$ (%)"
    )
    table.location = "Results"
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b"
    ]

    # Cache loaded files so we don't re-read the same m1 file multiple times
    file_cache = {}

    rows = []
    for m1, m2 in m1m2_pairs:
        if m1 not in file_cache:
            fpath = f"NPS25003_inputs/{subfolder}/higgsCombine_a1a2_{decay}_allchannels_allyears_m1_{m1}_limits.txt"
            file_cache[m1] = np.loadtxt(fpath, skiprows=1)
        data = file_cache[m1]
        if data.ndim == 1:
            data = data[np.newaxis, :]  # single-row file edge case
        # mh column (col 0) is m2 — find the matching row
        match = data[np.isclose(data[:, 0], m2)]
        if len(match) == 0:
            raise ValueError(f"m2={m2} not found in m1={m1} file for {decay}/{subfolder}")
        row = match[0]
        obs, exp_m2s, exp_m1s, exp_med, exp_p1s, exp_p2s = (
            row[1], row[2], row[3], row[4], row[5], row[6]
        )
        rows.append((m1, m2, obs, exp_m2s, exp_m1s, exp_med, exp_p1s, exp_p2s))

    # Independent variable: (m1, m2) mass pair as a string label
    mass_pair_var = Variable(
        "(phi_1 mass, phi_2 mass)",
        is_independent=True,
        is_binned=False,
        units="GeV"
    )
    mass_pair_var.values = [f"({m1}, {m2})" for m1, m2, *_ in rows]

    var_observed = Variable("Observed limit",             is_independent=False, is_binned=False, units="pb")
    var_exp_med  = Variable("Expected limit (median)",    is_independent=False, is_binned=False, units="pb")
    var_exp_m1s  = Variable("Expected limit (-1 sigma)",  is_independent=False, is_binned=False, units="pb")
    var_exp_p1s  = Variable("Expected limit (+1 sigma)",  is_independent=False, is_binned=False, units="pb")
    var_exp_m2s  = Variable("Expected limit (-2 sigma)",  is_independent=False, is_binned=False, units="pb")
    var_exp_p2s  = Variable("Expected limit (+2 sigma)",  is_independent=False, is_binned=False, units="pb")

    for v in (var_observed, var_exp_med, var_exp_m1s, var_exp_p1s, var_exp_m2s, var_exp_p2s):
        v.add_qualifier("SQRT(S)", "13", "TeV")

    var_observed.values = [float(r[2]) * XS_H for r in rows]
    var_exp_m2s.values  = [float(r[3]) * XS_H for r in rows]
    var_exp_m1s.values  = [float(r[4]) * XS_H for r in rows]
    var_exp_med.values  = [float(r[5]) * XS_H for r in rows]
    var_exp_p1s.values  = [float(r[6]) * XS_H for r in rows]
    var_exp_p2s.values  = [float(r[7]) * XS_H for r in rows]

    table.add_variable(mass_pair_var)
    table.add_variable(var_observed)
    table.add_variable(var_exp_med)
    table.add_variable(var_exp_m1s)
    table.add_variable(var_exp_p1s)
    table.add_variable(var_exp_m2s)
    table.add_variable(var_exp_p2s)

    table.add_image(f"NPS25003_inputs/NPS-25-003/{imageName}")

    print(table.name)
    return table

In [ ]:
import ROOT
ROOT.gROOT.SetBatch(True)
from hepdata_lib import Uncertainty
import numpy as np
import sys as _sys

# Reuse the exact helper/config code that produces the AN's official stat+shape
# background-uncertainty band, instead of reading a pre-harvested txt dump.
# This is the same dataMCPlots framework used by stackPlots_allyears.py.
_DATAMCPLOTS_DIR = "/afs/cern.ch/work/a/aquinn/public/CMSSW_13_1_0_pre4/src/lunaFramework/dataMCPlots"
if _DATAMCPLOTS_DIR not in _sys.path:
    _sys.path.insert(0, _DATAMCPLOTS_DIR)

from config.cardConfig import allHistsGroups, allShapeSystematics
from helpers.addSysFromShape import addAllSystematicsForYear
from helpers.getHists import getHistogram, getHistogramSum

_YEARS = ["2016preVFP", "2016postVFP", "2017", "2018"]

# One inclusive-category ROOT file per era per channel. Each file already contains
# every process (data, backgrounds, signal) and all shape-systematic variations
# for that era/channel, so all of Figures 2, 3, B.1, B.2 are built from these
# 12 files alone -- no intermediate txt harvesting.
_NEW_ROOT_BASE = "/eos/cms/store/group/phys_susy/AN-24-166/pdas/for_datacards"
_YEAR_FILES = {
    "mutau": [
        f"{_NEW_ROOT_BASE}/2026-08-14-23h40m_2016preVFP_signal2b2t_new_mt_inclusive/out_mutau.root",
        f"{_NEW_ROOT_BASE}/2026-08-14-23h38m_2016postVFP_signal2b2t_new_mt_inclusive/out_mutau.root",
        f"{_NEW_ROOT_BASE}/2026-08-14-23h36m_2017_signal2b2t_new_mt_inclusive/out_mutau.root",
        f"{_NEW_ROOT_BASE}/2026-08-14-23h34m_2018_signal2b2t_new_mt_inclusive/out_mutau.root",
    ],
    "etau": [
        f"{_NEW_ROOT_BASE}/2026-08-15-01h26m_2016preVFP_signal2b2t_new_et_inclusive/out_etau.root",
        f"{_NEW_ROOT_BASE}/2026-08-15-01h24m_2016postVFP_signal2b2t_new_et_inclusive/out_etau.root",
        f"{_NEW_ROOT_BASE}/2026-08-15-01h23m_2017_signal2b2t_new_et_inclusive/out_etau.root",
        f"{_NEW_ROOT_BASE}/2026-08-15-01h22m_2018_signal2b2t_new_et_inclusive/out_etau.root",
    ],
    "emu": [
        f"{_NEW_ROOT_BASE}/2026-08-15-02h44m_2016preVFP_signal2b2t_new_em_inclusive/out_emu.root",
        f"{_NEW_ROOT_BASE}/2026-08-15-02h44m_2016postVFP_signal2b2t_new_em_inclusive/out_emu.root",
        f"{_NEW_ROOT_BASE}/2026-08-15-02h43m_2017_signal2b2t_new_em_inclusive/out_emu.root",
        f"{_NEW_ROOT_BASE}/2026-08-15-03h03m_2018_signal2b2t_new_em_inclusive/out_emu.root",
    ],
}


def makePrefitDistTable(tableName, varSuffix, xLabel, xUnits, imageName, signal_samples, description, channel="mutau"):
    """
    Creates a HEPData Table for one pre-fit distribution subplot,
    summing inclusive histograms across all Run 2 years.

    Central values (data/background/signal) and the combined stat+shape background
    uncertainty are both derived in-notebook from the same 12 "inclusive" ROOT files
    (one per era per channel). Underflow/overflow are folded into the first/last bin
    for every histogram (matching the figure caption and the uncertainty algorithm
    below), via the shared getHistogram()/getHistogramSum() helpers.

    Parameters
    ----------
    tableName      : str  e.g. "Fig_002-a"
    varSuffix      : str  ROOT histogram key suffix, e.g. "D_zeta"
    xLabel         : str  x-axis label for HEPData
    xUnits         : str  x-axis units, e.g. "GeV" or ""
    imageName      : str  filename under NPS25003_inputs/NPS-25-003/
    signal_samples : list of ([proc_names], label)
                     e.g. [(["ggh4b2t-70-15", "vbf4b2t-70-15"], "cascade (15,70) GeV"), ...]
                     Histograms for all proc_names in each list are summed together.
    description    : str  table description
    channel        : str  "mutau", "etau", or "emu"
    """
    year_files = _YEAR_FILES[channel]
    category = "inclusive"

    SM_HIGGS_PROCS = [
        "ggh_htt", "ggh_hww", "qqh_htt", "qqh_hww",
        "Zh_htt", "Zh_hww", "Wh_htt", "Wh_hww", "tth",
    ]

    fakeName = "qcd" if channel == "emu" else "fake"
    if channel == "emu":
        INDIV_BG_PROCS = ["embedded", fakeName, "WJ", "ZJ", "ttbar", "ST", "VV"]
    else:
        INDIV_BG_PROCS = ["embedded", fakeName, "ZJ", "ttbar", "ST", "VV"]

    # "Other" background composition used both for the central-value sum and for
    # the per-era total-MC used in the uncertainty band, matching stackPlots_allyears.py.
    otherlist = ["ST", "VV"] + SM_HIGGS_PROCS + (["ZJ", "WJ"] if channel == "emu" else ["ZJ"])

    channel_label = {"mutau": "mu tau", "etau": "e tau", "emu": "e mu"}[channel]

    # ---- Central values: sum data/background/signal histograms across eras ----
    hists = {}

    def _add(key, h):
        if h is None:
            return
        if key not in hists:
            hists[key] = h.Clone()
            hists[key].SetDirectory(0)
        else:
            hists[key].Add(h)

    for fpath in year_files:
        f = ROOT.TFile.Open(fpath)
        if not f or f.IsZombie():
            print(f"WARNING: could not open {fpath}")
            continue

        _add("data_obs", getHistogram(f, "data_obs", varSuffix, channel, category))

        for proc in INDIV_BG_PROCS:
            _add(proc, getHistogram(f, proc, varSuffix, channel, category))

        _add("SM_Higgs", getHistogramSum(f, SM_HIGGS_PROCS, varSuffix, channel, "", category))

        for proc_names, _ in signal_samples:
            key = proc_names[0]
            _add(key, getHistogramSum(f, proc_names, varSuffix, channel, "", category))

        f.Close()

    # "Other" for mutau/etau = Z/gamma*->ee/mumu (ZJ), single top (ST), diboson (VV), SM Higgs
    # "Other" for emu = W+jets (WJ), ZJ, ST, VV, SM Higgs
    def _sum_hists(keys):
        total = None
        for k in keys:
            h = hists.get(k)
            if h is None:
                continue
            if total is None:
                total = h.Clone()
            else:
                total.Add(h)
        return total

    if channel == "emu":
        grouped_bkg = [
            (r"$Z\to\tau\tau$",  _sum_hists(["embedded"])),
            ("QCD",              _sum_hists([fakeName])),
            (r"$t\bar{t}$+jets", _sum_hists(["ttbar"])),
            ("Other",            _sum_hists(["WJ", "ZJ", "ST", "VV", "SM_Higgs"])),
        ]
    else:
        grouped_bkg = [
            (r"$Z\to\tau\tau$",  _sum_hists(["embedded"])),
            (r"Jet$\to\tau_h$",  _sum_hists([fakeName])),
            (r"$t\bar{t}$+jets", _sum_hists(["ttbar"])),
            ("Other",            _sum_hists(["ZJ", "ST", "VV", "SM_Higgs"])),
        ]

    hists["total_bkg"] = _sum_hists(INDIV_BG_PROCS + ["SM_Higgs"])

    # ---- Combined stat+shape background uncertainty, computed per era and summed
    # in quadrature across eras (exactly the algorithm in stackPlots_allyears.py:
    # addAllSystematicsForYear on a TGraphAsymmErrors seeded with totalMC's own
    # (Sumw2) stat error, then quadrature-summed across the 4 eras). ----
    totalErrUp2 = totalErrDown2 = None
    for year, fpath in zip(_YEARS, year_files):
        f = ROOT.TFile.Open(fpath)
        if not f or f.IsZombie():
            continue

        fakeHisto = getHistogram(f, fakeName, varSuffix, channel, category)
        embed = getHistogram(f, "embedded", varSuffix, channel, category)
        ttBar = getHistogram(f, "ttbar", varSuffix, channel, category)
        other = getHistogramSum(f, otherlist, varSuffix, channel, "", category)

        totalMC = fakeHisto.Clone()
        for x in [embed, ttBar, other]:
            totalMC.Add(x)

        sysErrorGraph = ROOT.TGraphAsymmErrors(totalMC.Clone())
        addAllSystematicsForYear(
            f, sysErrorGraph,
            allShapeSystematics[year][channel],
            allHistsGroups[year][channel],
            varSuffix, channel, category,
        )

        n = sysErrorGraph.GetN()
        errUp = np.array([sysErrorGraph.GetErrorYhigh(i) for i in range(n)])
        errDown = np.array([sysErrorGraph.GetErrorYlow(i) for i in range(n)])

        if totalErrUp2 is None:
            totalErrUp2 = errUp ** 2
            totalErrDown2 = errDown ** 2
        else:
            totalErrUp2 += errUp ** 2
            totalErrDown2 += errDown ** 2
        f.Close()

    bkg_err_up = np.sqrt(totalErrUp2) if totalErrUp2 is not None else None
    bkg_err_down = np.sqrt(totalErrDown2) if totalErrDown2 is not None else None

    # ---- Build the HEPData table ----
    table = Table(tableName)
    table.description = description
    table.location = "Results"
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b",
    ]

    href = hists["data_obs"]
    nbins = href.GetNbinsX()
    bin_edges = [
        (href.GetXaxis().GetBinLowEdge(i), href.GetXaxis().GetBinUpEdge(i))
        for i in range(1, nbins + 1)
    ]

    x_var = Variable(xLabel, is_independent=True, is_binned=True, units=xUnits)
    x_var.values = bin_edges
    table.add_variable(x_var)

    def _dep_var(label, hist, vtype, with_unc=False, err_up=None, err_down=None):
        v = Variable(label, is_independent=False, is_binned=False, units="Events")
        v.add_qualifier("SQRT(S)", "13", "TeV")
        v.add_qualifier("channel", channel_label)
        v.add_qualifier("type", vtype)
        vals = [hist.GetBinContent(i) for i in range(1, nbins + 1)]
        v.values = vals
        if err_up is not None and err_down is not None:
            unc = Uncertainty("stat+syst", is_symmetric=False)
            unc.values = [(-d, u) for d, u in zip(err_down, err_up)]
            v.add_uncertainty(unc)
        elif with_unc:
            unc = Uncertainty("stat", is_symmetric=True)
            unc.values = [hist.GetBinError(i) for i in range(1, nbins + 1)]
            v.add_uncertainty(unc)
        return v

    # Data: Poisson stat uncertainty from GetBinError
    table.add_variable(_dep_var("Data", hists["data_obs"], "data", with_unc=True))
    # Total background: combined stat+shape from the in-notebook computation above
    table.add_variable(_dep_var("Total background", hists["total_bkg"], "total background",
                                err_up=bkg_err_up, err_down=bkg_err_down))
    # Background groups: no uncertainty (individual MC stat would be incomplete)
    for label, hist in grouped_bkg:
        if hist is not None:
            table.add_variable(_dep_var(label, hist, "background"))
    # Signal: ggF+VBF summed; no uncertainty
    for proc_names, label in signal_samples:
        key = proc_names[0]
        if hists.get(key):
            table.add_variable(_dep_var(label, hists[key], "signal"))

    table.add_image(f"NPS25003_inputs/NPS-25-003/{imageName}")
    print(table.name)
    return table


In [ ]:
def makePostfitDistTable(tableName, fit_path, channel, sr_label, imageName,
                         signal_samples, description, signal_files,
                         signal_sr_key=None):
    CHANNEL_CH = {"mutau": "ch1", "etau": "ch2", "emu": "ch3"}
    CUTBASED_SR_CH = {"SRL": "ch1", "SRM": "ch2", "SRH": "ch3"}
    BDT_SR_CH = {
        "mutau": {"SR1_1b": "ch1", "SR2_1b": "ch2", "SR3_1b": "ch3",
                  "SR4_1b": "ch4", "SR1_2b": "ch5", "SR2_2b": "ch6"},
        "etau":  {"SR1_1b": "ch1", "SR2_1b": "ch2", "SR3_1b": "ch3",
                  "SR4_1b": "ch4", "SR1_2b": "ch5", "SR2_2b": "ch6"},
        "emu":   {"SR1_1b": "ch1", "SR2_1b": "ch2", "SR3_1b": "ch3",
                  "SR1_2b": "ch4", "SR2_2b": "ch5"},
    }
    YEAR_CH = ["ch4", "ch3", "ch2", "ch1"]  # preVFP, postVFP, 2017, 2018
    FIT_DIR = "shapes_fit_b"

    channel_ch = CHANNEL_CH[channel]
    sr_ch = CUTBASED_SR_CH[sr_label] if sr_label in CUTBASED_SR_CH else BDT_SR_CH[channel][sr_label]
    channel_label = {"mutau": "mu tau", "etau": "e tau", "emu": "e mu"}[channel]
    fake_name = "qcd" if channel == "emu" else "fake"
    signal_dir = signal_sr_key if signal_sr_key is not None else sr_label

    tfile = ROOT.TFile.Open(fit_path)

    def _get(path):
        h = tfile.Get(path)
        if not h: return None
        h = h.Clone(); h.SetDirectory(0); return h

    def _add(hists, key, h):
        if h is None: return
        if key not in hists: hists[key] = h
        else: hists[key].Add(h)

    hists = {}
    data_y = data_err_up = data_err_down = None
    bin_edges = None
    nbins = None

    for year_ch in YEAR_CH:
        dirpath = f"{FIT_DIR}/{year_ch}_{channel_ch}_{sr_ch}"
        ref = _get(f"{dirpath}/total_background")
        if ref is None:
            print(f"WARNING: {dirpath} not found, skipping"); continue
        if bin_edges is None:
            nbins = ref.GetNbinsX()
            bin_edges = [(ref.GetXaxis().GetBinLowEdge(i),
                          ref.GetXaxis().GetBinUpEdge(i)) for i in range(1, nbins + 1)]
        for proc in ["embedded", fake_name, "ttbar", "others", "total_background"]:
            _add(hists, proc, _get(f"{dirpath}/{proc}"))
        g = tfile.Get(f"{dirpath}/data")
        if g:
            n = g.GetN()
            if data_y is None:
                data_y = [0.0] * n
                data_err_up = [0.0] * n
                data_err_down = [0.0] * n
            for i in range(n):
                data_y[i] += g.GetPointY(i)
                data_err_up[i]   = np.sqrt(data_err_up[i]**2   + g.GetErrorYhigh(i)**2)
                data_err_down[i] = np.sqrt(data_err_down[i]**2 + g.GetErrorYlow(i)**2)
    tfile.Close()

    for proc_names, _ in signal_samples:
        key = proc_names[0]
        for proc_name in proc_names:
            for fpath in signal_files:
                f = ROOT.TFile.Open(fpath)
                if not f or f.IsZombie():
                    print(f"WARNING: could not open {fpath}"); continue
                h = f.Get(f"{signal_dir}/{proc_name}")
                if h:
                    h = h.Clone(); h.SetDirectory(0)
                    _add(hists, key, h)
                else:
                    print(f"WARNING: {signal_dir}/{proc_name} not in {fpath}")
                f.Close()

    table = Table(tableName)
    table.description = description
    table.location = "Results"
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b",
    ]

    x_var = Variable(r"$m_{\tau\tau}$", is_independent=True, is_binned=True, units="GeV")
    x_var.values = bin_edges
    table.add_variable(x_var)

    def _dep_var(label, vals, vtype, err_up=None, err_down=None,
                 stat_err_up=None, stat_err_down=None):
        v = Variable(label, is_independent=False, is_binned=False, units="Events")
        v.add_qualifier("SQRT(S)", "13", "TeV")
        v.add_qualifier("channel", channel_label)
        v.add_qualifier("SR", sr_label)
        v.add_qualifier("type", vtype)
        v.values = vals
        if err_up is not None and err_down is not None:
            unc = Uncertainty("stat+syst", is_symmetric=False)
            unc.values = [(-d, u) for d, u in zip(err_down, err_up)]
            v.add_uncertainty(unc)
        if stat_err_up is not None and stat_err_down is not None:
            unc = Uncertainty("stat", is_symmetric=False)
            unc.values = [(-d, u) for d, u in zip(stat_err_down, stat_err_up)]
            v.add_uncertainty(unc)
        return v

    table.add_variable(_dep_var("Data", data_y, "data",
                                stat_err_up=data_err_up, stat_err_down=data_err_down))

    tbkg = hists["total_background"]
    # Post-fit background uncertainty: combine already propagates the full covariance
    # into total_background's own bin error (see helpers/postfitRatioError.py, which builds
    # the AN's ratio-panel band the same way: sigma = totalBkg.GetBinError()). Adding the 4
    # eras' total_background histograms via TH1.Add() above already combined these in
    # quadrature, so no external bkg_unc txt is needed.
    bkg_err = [tbkg.GetBinError(i) for i in range(1, nbins + 1)]
    table.add_variable(_dep_var("Total background",
                                [tbkg.GetBinContent(i) for i in range(1, nbins + 1)],
                                "total background", err_up=bkg_err, err_down=bkg_err))

    _bkg_labels = {
        "embedded": r"$Z\to\tau\tau$",
        "fake":     r"Jet$\to\tau_h$",
        "qcd":      "QCD",
        "ttbar":    r"$t\bar{t}$+jets",
        "others":   "Other",
    }
    for proc in ["embedded", fake_name, "ttbar", "others"]:
        h = hists.get(proc)
        if h:
            table.add_variable(_dep_var(_bkg_labels[proc],
                                        [h.GetBinContent(i) for i in range(1, nbins + 1)],
                                        "background"))

    # Signal: ggF+VBF summed; no uncertainty
    for proc_names, label in signal_samples:
        key = proc_names[0]
        h = hists.get(key)
        if h:
            table.add_variable(_dep_var(label,
                                        [h.GetBinContent(i) for i in range(1, nbins + 1)],
                                        "signal"))

    table.add_image(f"NPS25003_inputs/NPS-25-003/{imageName}")
    print(table.name)
    return table

In [ ]:
def makeEfficiencyTable(tableName, entries, sr_description=None, channel=None):
    """
    Creates a HEPData Table for signal selection efficiencies displayed as a 2D map.

    Parameters
    ----------
    tableName : str
    entries   : list of (m1, m2, efficiency)
                For mass points evaluated in both topologies, pass the non-cascade
                value (consistent with the 2D limit plot convention).
    channel   : str or None  "mutau", "etau", or "emu"
    """
    _channel_labels = {
        "mutau": r"$\mu\tau_h$",
        "etau":  r"$e\tau_h$",
        "emu":   r"$e\mu$",
    }
    channel_label = _channel_labels.get(channel) if channel else None
    _prefix = (
        ("For the " + channel_label + " channel" if channel_label else "")
        + (" in " + sr_description if sr_description else "")
        + (": " if (channel_label or sr_description) else "")
    )
    table = Table(tableName)
    table.description = (
        _prefix
        + r"Signal selection efficiency for simplified model mass points as a function"
        r" of the scalar masses $m_{\phi_1}$ and $m_{\phi_2}$"
        + r". For the mass"
        r" hypotheses $(m_{\phi_1}, m_{\phi_2})$ = (15, 30), (20, 40), and (30, 60) GeV,"
        r" evaluated in both the cascade ($\mathrm{H} \to \phi_1 \phi_2 \to 2\tau\,4b$)"
        r" and non-cascade ($\mathrm{H} \to \phi_1 \phi_2 \to 2\tau\,2b$) scenarios,"
        r" only the non-cascade efficiency is shown, consistent with the convention"
        r" used in Figure 9."
    )
    table.location = "Auxiliary material"
    table.keywords["observables"] = ["EFF"]
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b",
    ]

    phi1_var = Variable(
        r"$m_{\phi_1}$",
        is_independent=True,
        is_binned=False,
        units="GeV",
    )
    phi1_var.values = [m1 for m1, m2, e in entries]

    phi2_var = Variable(
        r"$m_{\phi_2}$",
        is_independent=True,
        is_binned=False,
        units="GeV",
    )
    phi2_var.values = [m2 for m1, m2, e in entries]

    eff_var = Variable(
        "Signal efficiency",
        is_independent=False,
        is_binned=False,
        units="",
    )
    eff_var.values = [float(e) for m1, m2, e in entries]
    eff_var.add_qualifier("SQRT(S)", "13", "TeV")

    table.add_variable(phi1_var)
    table.add_variable(phi2_var)
    table.add_variable(eff_var)
    print(table.name)
    return table

## Main Function

The `Submission` object represents the whole HEPData entry and thus carries the top-level meta data that is equally valid for all the tables and variables you may want to enter. The object is also used to create the physical submission files you will upload to the HEPData web interface.

When using `hepdata_lib` to make an entry, you always need to create a `Submission` object. 

In [ ]:
def main():
    submission = Submission()
    submission.read_abstract("NPS25003_inputs/abstract.txt")

    #ADL
    submission.add_additional_resource("ADL file", "NPS25003_inputs/NPS25003.adl", copy_file=True)

    #Production cross section
    submission.add_link("Standard ggF and VBF cross-sections from Handbook of LHC Cross-sections", "http://arxiv.org/abs/arXiv:1610.07922") 

    #Signal model UFO Files
    submission.add_link("Signal Model UFO files", "https://gitlab.com/apapaefs/twosinglet")

    #Generator Process cards
    submission.add_link("Generator Process Cards", "https://github.com/cms-sw/genproductions/pull/2705") 

    #Small set of input vectors & ML outputs
    submission.add_link("BDT Models", "https://github.com/Aaravind96/aabbttBDT/tree/preservation/BDTmodels")

    #Statistical model
    submission.add_link("Datacards", "https://gitlab.cern.ch/cms-analysis/nps/nps-25-003/datacards/-/tree/master/input?ref_type=heads") 
    
    #Final wiki
    submission.add_link("Wiki", "https://cms-results.web.cern.ch/cms-results/public-results/publications/NPS-25-003/") 

    version_folder = "v11_limits"
    plots = "NPS-25-003"

    ##############################
    # Pre-fit distributions      #
    # Figure 002 (mutau channel) #
    ##############################

    # Process name format: {mode}{decay}-{m2}-{m1}
    #   mode:  "ggh" (ggF) or "vbf"
    #   decay: "4b2t" (cascade) or "2b2t" (non-cascade)
    #   e.g. "ggh4b2t-70-15" = ggF cascade, m_phi2=70, m_phi1=15 GeV
    _prefit_signals = [
        (["ggh4b2t-70-15", "vbf4b2t-70-15"], r"cascade $(m_{\phi_1},m_{\phi_2})=(15,70)$ GeV (ggF+VBF)"),
        (["ggh2b2t-30-20", "vbf2b2t-30-20"], r"non-cascade $(m_{\phi_1},m_{\phi_2})=(20,30)$ GeV (ggF+VBF)"),
        (["ggh4b2t-80-30", "vbf4b2t-80-30"], r"cascade $(m_{\phi_1},m_{\phi_2})=(30,80)$ GeV (ggF+VBF)"),
        (["ggh2b2t-60-40", "vbf2b2t-60-40"], r"non-cascade $(m_{\phi_1},m_{\phi_2})=(40,60)$ GeV (ggF+VBF)"),
    ]

    _prefit_desc = (
        r"Pre-fit distributions of {var}, including underflow and overflow bins,"
        r" for preselected events with at least one b-tagged jet for the $\mu\tau_h$ channel,"
        r" without any SR requirements. The data are shown by the markers with vertical bars"
        r" and various backgrounds by the colored histograms. The combination of statistical"
        r" and shape systematic uncertainties is displayed with the hatched areas. The colored"
        r" open histograms display the predicted signal distribution for two cascade decays and"
        r" two non-cascade decays, with four different values of $\phi_1$ and $\phi_2$ masses,"
        r" for an assumed branching fraction of 100%. The lower panel of each plot shows the"
        r" ratio of the data to the sum of the predicted number of background events. The"
        r" vertical bars on the points show the statistical uncertainty in the ratio."
    )

    table_configs_prefit = [
        # (tableName,  varSuffix,        xLabel,                                                xUnits, imageName)
        ("Fig_002-a", "D_zeta",        r"$D_\zeta$",                                          "GeV",  "Figure_002-a.pdf"),
        ("Fig_002-b", "m_btautau_vis", r"$m^\mathrm{vis}(\tau\tau b_1)$",                     "GeV",  "Figure_002-b.pdf"),
        ("Fig_002-c", "mtMET_1",       r"$m_\mathrm{T}(\mu, p_\mathrm{T}^\mathrm{miss})$",    "GeV",  "Figure_002-c.pdf"),
        ("Fig_002-d", "mtMET_2",       r"$m_\mathrm{T}(\tau_h, p_\mathrm{T}^\mathrm{miss})$", "GeV",  "Figure_002-d.pdf"),
    ]

    for tableName, varSuffix, xLabel, xUnits, imageName in table_configs_prefit:
        submission.add_table(makePrefitDistTable(
            tableName, varSuffix, xLabel, xUnits, imageName,
            _prefit_signals,
            _prefit_desc.format(var=xLabel),
        ))


        ######################################
    # Pre-fit BDT score distributions    #
    # Figure 003 (mutau, etau, emu)      #
    ######################################

    _fig3_desc = (
        r"Pre-fit BDT score distribution for preselected events with at least one"
        r" b-tagged jet for the {channel_label} channel, without any SR requirements."
        r" Data are shown by the markers with vertical bars and various backgrounds by"
        r" the colored histograms. The combination of statistical and shape systematic"
        r" uncertainties is displayed with the hatched areas. The colored open histograms"
        r" display the predicted signal distribution for two cascade decays and two"
        r" non-cascade decays, with four different values of $m_{{\phi_1}}$ and"
        r" $m_{{\phi_2}}$ masses, for an assumed branching fraction of 100%."
        r" The lower panel shows the ratio of the data to the sum of the predicted"
        r" number of background events. The vertical bars on the points show the"
        r" statistical uncertainty in the ratio."
    )

    _channel_labels = {
        "mutau": r"$\mu\tau_h$",
        "etau":  r"$e\tau_h$",
        "emu":   r"$e\mu$",
    }

    table_configs_fig3 = [
        # (tableName,  channel,  imageName)
        ("Fig_003-a", "mutau", "Figure_003-a.pdf"),
        ("Fig_003-b", "etau",  "Figure_003-b.pdf"),
        ("Fig_003-c", "emu",   "Figure_003-c.pdf"),
    ]

    for tableName, channel, imageName in table_configs_fig3:
        submission.add_table(makePrefitDistTable(
            tableName, "bdtscore", "BDT score", "", imageName,
            _prefit_signals,
            _fig3_desc.format(channel_label=_channel_labels[channel]),
            channel=channel,
        ))

    ############################################
    # Post-fit m_tt distributions (BDT-based)  #
    # Figures 004 (mutau), 005 (etau), 006 (emu)
    ############################################

    _FIT_PATH_BDT  = "NPS25003_inputs/fitDiagnostics_60_40_bdtbased_postfit_try5.root"
    _FIT_PATH_CUT  = "NPS25003_inputs/fitDiagnostics_60_40_cutbased_postfit_try5.root"

    # Signal shapes: SR/CR-categorized datacard shape inputs (pre-fit templates).
    # Same file family also supplies "other_aux_new" background/data used to validate
    # that the (60,40) fitDiagnostics files above remain valid: only the signal grid
    # changed in this reprocessing (added the phi1->2tau non-cascade orientation),
    # backgrounds and data are bit-identical to what produced those fits.
    _OTHER_AUX_NEW = "/eos/cms/store/group/phys_susy/AN-24-166/pdas/BACKUP_asymmCards/other_aux_new"
    _BDT_SIGNAL_FILES = {
        "mutau": [
            f"{_OTHER_AUX_NEW}/bdt/shapes2016preVFP/out_mutau.root",
            f"{_OTHER_AUX_NEW}/bdt/shapes2016postVFP/out_mutau.root",
            f"{_OTHER_AUX_NEW}/bdt/shapes2017/out_mutau.root",
            f"{_OTHER_AUX_NEW}/bdt/shapes2018/out_mutau.root",
        ],
        "etau": [
            f"{_OTHER_AUX_NEW}/bdt/shapes2016preVFP/out_etau.root",
            f"{_OTHER_AUX_NEW}/bdt/shapes2016postVFP/out_etau.root",
            f"{_OTHER_AUX_NEW}/bdt/shapes2017/out_etau.root",
            f"{_OTHER_AUX_NEW}/bdt/shapes2018/out_etau.root",
        ],
        "emu": [
            f"{_OTHER_AUX_NEW}/bdt/shapes2016preVFP/out_emu.root",
            f"{_OTHER_AUX_NEW}/bdt/shapes2016postVFP/out_emu.root",
            f"{_OTHER_AUX_NEW}/bdt/shapes2017/out_emu.root",
            f"{_OTHER_AUX_NEW}/bdt/shapes2018/out_emu.root",
        ],
    }

    _CUT_SIGNAL_FILES = {
        "mutau": [
            f"{_OTHER_AUX_NEW}/shapes2016preVFP/out_mutau.root",
            f"{_OTHER_AUX_NEW}/shapes2016postVFP/out_mutau.root",
            f"{_OTHER_AUX_NEW}/shapes2017/out_mutau.root",
            f"{_OTHER_AUX_NEW}/shapes2018/out_mutau.root",
        ],
        "etau": [
            f"{_OTHER_AUX_NEW}/shapes2016preVFP/out_etau.root",
            f"{_OTHER_AUX_NEW}/shapes2016postVFP/out_etau.root",
            f"{_OTHER_AUX_NEW}/shapes2017/out_etau.root",
            f"{_OTHER_AUX_NEW}/shapes2018/out_etau.root",
        ],
        "emu": [
            f"{_OTHER_AUX_NEW}/shapes2016preVFP/out_emu.root",
            f"{_OTHER_AUX_NEW}/shapes2016postVFP/out_emu.root",
            f"{_OTHER_AUX_NEW}/shapes2017/out_emu.root",
            f"{_OTHER_AUX_NEW}/shapes2018/out_emu.root",
        ],
    }

    _postfit_signals = _prefit_signals  # same 8 signal samples

    _bdt_channel_labels = {
        "mutau": r"$\mu\tau_h$",
        "etau":  r"$e\tau_h$",
        "emu":   r"$e\mu$",
    }

    # BDT SR layouts per channel for the figure caption
    _bdt_sr_layout = {
        "mutau": (
            "in events with exactly one b-tagged jet: SR1 (upper left), SR2 (upper right), "
            "SR3 (middle left), and SR4 (middle right), and in events with at least two "
            "b-tagged jets: SR1 (lower left) and SR2 (lower right)"
        ),
        "etau": (
            "in events with exactly one b-tagged jet: SR1 (upper left), SR2 (upper right), "
            "SR3 (middle left), and SR4 (middle right), and in events with at least two "
            "b-tagged jets: SR1 (lower left) and SR2 (lower right)"
        ),
        "emu": (
            "in events with exactly one b-tagged jet: SR1 (upper left), SR2 (upper right), "
            "and SR3 (lower left), and in events with at least two b-tagged jets: "
            "SR1 (lower center) and SR2 (lower right)"
        ),
    }

    _bdt_table_configs = [
        # Figure 004 — mutau
        ("Fig_004-a", "mutau", "SR1_1b", "Figure_004-a.pdf"),
        ("Fig_004-b", "mutau", "SR2_1b", "Figure_004-b.pdf"),
        ("Fig_004-c", "mutau", "SR3_1b", "Figure_004-c.pdf"),
        ("Fig_004-d", "mutau", "SR4_1b", "Figure_004-d.pdf"),
        ("Fig_004-e", "mutau", "SR1_2b", "Figure_004-e.pdf"),
        ("Fig_004-f", "mutau", "SR2_2b", "Figure_004-f.pdf"),
        # Figure 005 — etau
        ("Fig_005-a", "etau", "SR1_1b", "Figure_005-a.pdf"),
        ("Fig_005-b", "etau", "SR2_1b", "Figure_005-b.pdf"),
        ("Fig_005-c", "etau", "SR3_1b", "Figure_005-c.pdf"),
        ("Fig_005-d", "etau", "SR4_1b", "Figure_005-d.pdf"),
        ("Fig_005-e", "etau", "SR1_2b", "Figure_005-e.pdf"),
        ("Fig_005-f", "etau", "SR2_2b", "Figure_005-f.pdf"),
        # Figure 006 — emu
        ("Fig_006-a", "emu", "SR1_1b", "Figure_006-a.pdf"),
        ("Fig_006-b", "emu", "SR2_1b", "Figure_006-b.pdf"),
        ("Fig_006-c", "emu", "SR3_1b", "Figure_006-c.pdf"),
        ("Fig_006-d", "emu", "SR1_2b", "Figure_006-d.pdf"),
        ("Fig_006-e", "emu", "SR2_2b", "Figure_006-e.pdf"),
    ]

    for tableName, channel, sr_label, imageName in _bdt_table_configs:
        _chan_label = _bdt_channel_labels[channel]
        _desc = (
            r"Background only, post-fit $m_{\tau\tau}$ distributions for the "
            + _chan_label + r" channel, "
            + _bdt_sr_layout[channel]
            + r". The data are shown by the markers with vertical bars and various"
            r" backgrounds by the colored histograms. The total systematic uncertainty"
            r" is shown by the hatched area. The colored open histograms display the"
            r" predicted signal distribution for two cascade decays and two non-cascade"
            r" decays, with four different values of $\phi_1$ and $\phi_2$ masses,"
            r" for an assumed branching fraction of 100%. The lower plot of each"
            r" panel gives the ratio of the data to the sum of the predicted number of"
            r" background events. The vertical bars display the statistical uncertainty"
            r" in the ratio. This table corresponds to " + sr_label + "."
        )
        submission.add_table(makePostfitDistTable(
            tableName, _FIT_PATH_BDT, channel, sr_label, imageName,
            _postfit_signals, _desc,
            _BDT_SIGNAL_FILES[channel],
        ))

    ######################
    # 1D Limit plots     #
    # Figures 007, 008   #
    # (BDT-based)        #
    ######################

    _cascade_pairs = [
        (15,30),(15,40),(15,50),(15,60),(15,70),(15,80),(15,90),(15,100),(15,110),
        (20,40),(20,50),(20,60),(20,70),(20,80),(20,90),(20,100),
        (30,60),(30,70),(30,80),(30,90),
    ]
    _noncascade_pairs = [
        (15,20),(15,30),(20,30),(20,40),(30,40),(30,50),(30,60),
        (40,50),(40,60),(40,70),(40,80),(50,60),(50,70),
    ]

    descriptions_1D = {
        "Fig_007": (
            r"The observed (points) and median expected (dotted line) 95% CL upper limits on"
            r" $\sigma B_\mathrm{C}$ for the cascade scenario using the BDT-based event"
            r" categorization and the fit to the $m_{\tau\tau}$ distribution, for different"
            r" mass hypotheses $(m_{\phi_1}, m_{\phi_2})$. The horizontal bars on the points"
            r" are for better legibility only. The green and yellow regions show the 68 and 95%"
            r" expected range for the median value, respectively."
        ),
        "Fig_008": (
            r"The observed (points) and median expected (dotted line) 95% CL upper limits on"
            r" $\sigma B_\mathrm{NC}$ for the non-cascade scenario using the BDT-based event"
            r" categorization and the fit to the $m_{\tau\tau}$ distribution, for different"
            r" mass hypotheses $(m_{\phi_1}, m_{\phi_2})$. The horizontal bars on the points"
            r" are for better legibility only. The green and yellow regions show the 68 and 95%"
            r" expected range for the median value, respectively."
        ),
        "Fig_A.4": (
            r"The observed (points) and median expected (dotted line) 95% CL upper limits on"
            r" $\sigma B_\mathrm{C}$ for the cascade scenario using the cut-based event"
            r" categorization and the fit to the $m_{\tau\tau}$ distribution, for different"
            r" mass hypotheses $(m_{\phi_1}, m_{\phi_2})$. The horizontal bars on the points"
            r" are for better legibility only. The green and yellow regions show the 68 and 95%"
            r" expected range for the median value, respectively."
        ),
        "Fig_A.5": (
            r"The observed (points) and median expected (dotted line) 95% CL upper limits on"
            r" $\sigma B_\mathrm{NC}$ for the non-cascade scenario using the cut-based event"
            r" categorization and the fit to the $m_{\tau\tau}$ distribution, for different"
            r" mass hypotheses $(m_{\phi_1}, m_{\phi_2})$. The horizontal bars on the points"
            r" are for better legibility only. The green and yellow regions show the 68 and 95%"
            r" expected range for the median value, respectively."
        ),
    }

    for tableName, subfolder, decay, isBDT, isCascade, pairs, imageName in [
        ("Fig_007", "bdtbased_root", "4b2t", True,  True,  _cascade_pairs,    "Figure_007.pdf"),
        ("Fig_008", "bdtbased_root", "2b2t", True,  False, _noncascade_pairs, "Figure_008.pdf"),
    ]:
        submission.add_table(
            make1DLimitTable_v13(tableName, pairs, subfolder, decay, isBDT, isCascade, imageName,
                                 description=descriptions_1D[tableName])
        )

    ##################
    # 2D Limit plots #
    # Figure 009     #
    # (BDT-based)    #
    ##################

    for name, isBDT, subfolder, channel, imageName in [
        ("Fig_009-a_2D_BDT_mutau",       True, "bdt_based", "mutau",       f"{plots}/Figure_009-a.pdf"),
        ("Fig_009-b_2D_BDT_etau",        True, "bdt_based", "etau",        f"{plots}/Figure_009-b.pdf"),
        ("Fig_009-c_2D_BDT_emu",         True, "bdt_based", "emu",         f"{plots}/Figure_009-c.pdf"),
        ("Fig_009-d_2D_BDT_allchannels", True, "bdt_based", "allchannels", f"{plots}/Figure_009-d.pdf"),
    ]:
        submission.add_table(make2DLimitTable(
            name, isBDT,
            f"{version_folder}/{subfolder}/median_limits_{channel}.txt",
            imageName,
        ))

    #################################################
    # Post-fit m_tt distributions (cut-based)        #
    # Figures A.1 (mutau), A.2 (etau), A.3 (emu)     #
    #################################################

    _cut_sr_layout = (
        "in the low-mass SR (left), medium-mass SR (center), and high-mass SR (right)"
    )
    _CUT_SIGNAL_SR_KEY = {"SRL": "lowMassSR", "SRM": "mediumMassSR", "SRH": "highMassSR"}

    _cut_table_configs = [
        # Figure A.1 — mutau
        ("Fig_A.1-a", "mutau", "SRL", "Figure_010-a.pdf"),
        ("Fig_A.1-b", "mutau", "SRM", "Figure_010-b.pdf"),
        ("Fig_A.1-c", "mutau", "SRH", "Figure_010-c.pdf"),
        # Figure A.2 — etau
        ("Fig_A.2-a", "etau", "SRL", "Figure_011-a.pdf"),
        ("Fig_A.2-b", "etau", "SRM", "Figure_011-b.pdf"),
        ("Fig_A.2-c", "etau", "SRH", "Figure_011-c.pdf"),
        # Figure A.3 — emu
        ("Fig_A.3-a", "emu", "SRL", "Figure_012-a.pdf"),
        ("Fig_A.3-b", "emu", "SRM", "Figure_012-b.pdf"),
        ("Fig_A.3-c", "emu", "SRH", "Figure_012-c.pdf"),
    ]

    for tableName, channel, sr_label, imageName in _cut_table_configs:
        _chan_label = _bdt_channel_labels[channel]
        _desc = (
            r"Background only, post-fit $m_{\tau\tau}$ distributions for the "
            + _chan_label + r" channel, "
            + _cut_sr_layout
            + r". The data are shown by the markers with vertical bars and various"
            r" backgrounds by the colored histograms. The total systematic uncertainty"
            r" is shown by the hatched area. The colored open histograms display the"
            r" predicted signal distribution for two cascade decays and two non-cascade"
            r" decays, with four different values of $\phi_1$ and $\phi_2$ masses,"
            r" for an assumed branching fraction of 100%. The lower plot of each"
            r" panel gives the ratio of the data to the sum of the predicted number of"
            r" background events. The vertical bars display the statistical uncertainty"
            r" in the ratio. This table corresponds to " + sr_label + "."
        )
        submission.add_table(makePostfitDistTable(
            tableName, _FIT_PATH_CUT, channel, sr_label, imageName,
            _postfit_signals, _desc,
            _CUT_SIGNAL_FILES[channel],
            signal_sr_key=_CUT_SIGNAL_SR_KEY[sr_label],
        ))

    ######################
    # 1D Limit plots     #
    # Figures A.4, A.5   #
    # (cut-based)        #
    ######################

    for tableName, subfolder, decay, isBDT, isCascade, pairs, imageName in [
        ("Fig_A.4", "cutbased_root", "4b2t", False, True,  _cascade_pairs,    "Figure_013.pdf"),
        ("Fig_A.5", "cutbased_root", "2b2t", False, False, _noncascade_pairs, "Figure_014.pdf"),
    ]:
        submission.add_table(
            make1DLimitTable_v13(tableName, pairs, subfolder, decay, isBDT, isCascade, imageName,
                                 description=descriptions_1D[tableName])
        )

    ##################
    # 2D Limit plots #
    # Figure A.6     #
    # (cut-based)    #
    ##################

    for name, isBDT, subfolder, channel, imageName in [
        ("Fig_A.6-a_2D_cutbased_mutau",       False, "cut_based", "mutau",       f"{plots}/Figure_015-a.pdf"),
        ("Fig_A.6-b_2D_cutbased_etau",        False, "cut_based", "etau",        f"{plots}/Figure_015-b.pdf"),
        ("Fig_A.6-c_2D_cutbased_emu",         False, "cut_based", "emu",         f"{plots}/Figure_015-c.pdf"),
        ("Fig_A.6-d_2D_cutbased_allchannels", False, "cut_based", "allchannels", f"{plots}/Figure_015-d.pdf"),
    ]:
        submission.add_table(make2DLimitTable(
            name, isBDT,
            f"{version_folder}/{subfolder}/median_limits_{channel}.txt",
            imageName,
        ))

    ##################################
    # Pre-fit distributions          #
    # Appendix B, Figure B.1 (etau) #
    ##################################

    _prefit_desc_etau = (
        r"Pre-fit distributions of {var}, including underflow and overflow bins,"
        r" for preselected events with at least one b-tagged jet for the $e\tau_h$ channel,"
        r" without any SR requirements. The data are shown by the markers with vertical bars"
        r" and various backgrounds by the colored histograms. The combination of statistical"
        r" and shape systematic uncertainties is displayed with the hatched areas. The colored"
        r" open histograms display the predicted signal distribution for two cascade decays and"
        r" two non-cascade decays, with four different values of $\phi_1$ and $\phi_2$ masses,"
        r" for an assumed branching fraction of 100%. The lower panel of each plot shows the"
        r" ratio of the data to the sum of the predicted number of background events. The"
        r" vertical bars on the points show the statistical uncertainty in the ratio."
    )

    table_configs_prefit_etau = [
        # (tableName,  varSuffix,        xLabel,                                                 xUnits, imageName)
        ("Fig_B.1-a", "D_zeta",        r"$D_\zeta$",                                           "GeV",  "Figure_016-a.pdf"),
        ("Fig_B.1-b", "m_btautau_vis", r"$m^\mathrm{vis}(\tau\tau b_1)$",                      "GeV",  "Figure_016-b.pdf"),
        ("Fig_B.1-c", "mtMET_1",       r"$m_\mathrm{T}(e, p_\mathrm{T}^\mathrm{miss})$",      "GeV",  "Figure_016-c.pdf"),
        ("Fig_B.1-d", "mtMET_2",       r"$m_\mathrm{T}(\tau_h, p_\mathrm{T}^\mathrm{miss})$", "GeV",  "Figure_016-d.pdf"),
    ]

    for tableName, varSuffix, xLabel, xUnits, imageName in table_configs_prefit_etau:
        submission.add_table(makePrefitDistTable(
            tableName, varSuffix, xLabel, xUnits, imageName,
            _prefit_signals,
            _prefit_desc_etau.format(var=xLabel),
            channel="etau",
        ))

    #################################
    # Pre-fit distributions         #
    # Appendix B, Figure B.2 (emu) #
    #################################

    _prefit_desc_emu = (
        r"Pre-fit distributions of {var}, including underflow and overflow bins,"
        r" for preselected events with at least one b-tagged jet for the $e\mu$ channel,"
        r" without any SR requirements. The data are shown by the markers with vertical bars"
        r" and various backgrounds by the colored histograms. The combination of statistical"
        r" and shape systematic uncertainties is displayed with the hatched areas. The colored"
        r" open histograms display the predicted signal distribution for two cascade decays and"
        r" two non-cascade decays, with four different values of $\phi_1$ and $\phi_2$ masses,"
        r" for an assumed branching fraction of 100%. The lower panel of each plot shows the"
        r" ratio of the data to the sum of the predicted number of background events. The"
        r" vertical bars on the points show the statistical uncertainty in the ratio."
    )

    table_configs_prefit_emu = [
        # (tableName,  varSuffix,        xLabel,                                               xUnits, imageName)
        ("Fig_B.2-a", "D_zeta",        r"$D_\zeta$",                                         "GeV",  "Figure_017-a.pdf"),
        ("Fig_B.2-b", "m_btautau_vis", r"$m^\mathrm{vis}(\tau\tau b_1)$",                    "GeV",  "Figure_017-b.pdf"),
        ("Fig_B.2-c", "mtMET_1",       r"$m_\mathrm{T}(\mu, p_\mathrm{T}^\mathrm{miss})$",  "GeV",  "Figure_017-c.pdf"),
        ("Fig_B.2-d", "mtMET_2",       r"$m_\mathrm{T}(e, p_\mathrm{T}^\mathrm{miss})$",    "GeV",  "Figure_017-d.pdf"),
    ]

    for tableName, varSuffix, xLabel, xUnits, imageName in table_configs_prefit_emu:
        submission.add_table(makePrefitDistTable(
            tableName, varSuffix, xLabel, xUnits, imageName,
            _prefit_signals,
            _prefit_desc_emu.format(var=xLabel),
            channel="emu",
        ))


    ####################
    # Cutflow          #
    ####################

    with open("NPS25003_inputs/cutflow.txt") as _cf_file:
        _cf_lines = [l.rstrip("\n") for l in _cf_file.readlines()]

    # Section 1: signal MC yields (lines 1-9, header at line 0)
    # Fixed-width columns: cut_name[53] | (60,40)[16] | (80,30)
    _sig_cuts, _sig_60_40, _sig_80_30 = [], [], []
    for _raw in _cf_lines[1:10]:
        _name = _raw[:53].strip()
        _vals = _raw[53:].split()
        _sig_cuts.append(_name)
        _sig_60_40.append(float(_vals[0]))
        _sig_80_30.append(float(_vals[1]))

    # Section 2: background yields (lines 12-19, header at line 11)
    # Fixed-width columns: cut_name[53] | ttbar[16] | Z->tautau[16] | QCD[16] | fakes
    _bkg_cuts, _bkg_ttbar, _bkg_ztautau, _bkg_qcd, _bkg_fake = [], [], [], [], []
    for _raw in _cf_lines[12:]:
        _name = _raw[:53].strip()
        _vals = _raw[53:].split()
        _bkg_cuts.append(_name)
        _bkg_ttbar.append(float(_vals[0]))
        _bkg_ztautau.append(float(_vals[1]))
        _bkg_qcd.append(float(_vals[2]) if _vals[2] != "-" else "-")
        _bkg_fake.append(float(_vals[3]))

    _bkg_idx = {c: i for i, c in enumerate(_bkg_cuts)}
    _all_cuts = _sig_cuts + [c for c in _bkg_cuts if c not in set(_sig_cuts)]

    _cf_sig_60_40, _cf_sig_80_30 = [], []
    _cf_ttbar, _cf_ztautau, _cf_qcd, _cf_fake = [], [], [], []
    for cut in _all_cuts:
        if cut in _sig_cuts:
            i = _sig_cuts.index(cut)
            _cf_sig_60_40.append(_sig_60_40[i])
            _cf_sig_80_30.append(_sig_80_30[i])
        else:
            _cf_sig_60_40.append("-")
            _cf_sig_80_30.append("-")
        if cut in _bkg_idx:
            i = _bkg_idx[cut]
            _cf_ttbar.append(_bkg_ttbar[i])
            _cf_ztautau.append(_bkg_ztautau[i])
            _cf_qcd.append(_bkg_qcd[i])
            _cf_fake.append(_bkg_fake[i])
        else:
            _cf_ttbar.append("-")
            _cf_ztautau.append("-")
            _cf_qcd.append("-")
            _cf_fake.append("-")

    table_cf = Table("Cutflow")
    table_cf.description = (
        r"Cutflow showing event yields after each selection step for two signal mass"
        r" points and the major backgrounds, for the $\mu\tau_h$, $e\tau_h$, and $e\mu$ channels."
        r" Signal mass points are labelled by $(m_{{\phi_1}}, m_{{\phi_2}})$ in GeV,"
        r" for an assumed branching fraction of 100%."
        r" Entries marked '-' are not applicable for that sample."
    )
    table_cf.location = "Auxiliary material"
    table_cf.keywords["observables"] = ["N"]

    _cuts_var = Variable("Selection step", is_independent=True, is_binned=False, units="")
    _cuts_var.values = _all_cuts
    table_cf.add_variable(_cuts_var)

    _var_sig1 = Variable("Signal yield", is_independent=False, is_binned=False, units="Events")
    _var_sig1.values = _cf_sig_60_40
    _var_sig1.add_qualifier(r"$(m_{{\phi_1}}, m_{{\phi_2}})$", "(60, 40) GeV")
    _var_sig1.add_qualifier("SQRT(S)", 13, "TeV")
    _var_sig1.add_qualifier("type", "signal")
    table_cf.add_variable(_var_sig1)

    _var_sig2 = Variable("Signal yield", is_independent=False, is_binned=False, units="Events")
    _var_sig2.values = _cf_sig_80_30
    _var_sig2.add_qualifier(r"$(m_{{\phi_1}}, m_{{\phi_2}})$", "(80, 30) GeV")
    _var_sig2.add_qualifier("SQRT(S)", 13, "TeV")
    _var_sig2.add_qualifier("type", "signal")
    table_cf.add_variable(_var_sig2)

    _var_ttbar = Variable(r"$t\bar{t}$+jets", is_independent=False, is_binned=False, units="Events")
    _var_ttbar.values = _cf_ttbar
    _var_ttbar.add_qualifier("SQRT(S)", 13, "TeV")
    _var_ttbar.add_qualifier("type", "background")
    table_cf.add_variable(_var_ttbar)

    _var_ztautau = Variable(r"$Z\to\tau\tau$", is_independent=False, is_binned=False, units="Events")
    _var_ztautau.values = _cf_ztautau
    _var_ztautau.add_qualifier("SQRT(S)", 13, "TeV")
    _var_ztautau.add_qualifier("type", "background")
    table_cf.add_variable(_var_ztautau)

    _var_qcd = Variable("QCD", is_independent=False, is_binned=False, units="Events")
    _var_qcd.values = _cf_qcd
    _var_qcd.add_qualifier("SQRT(S)", 13, "TeV")
    _var_qcd.add_qualifier("type", "background")
    table_cf.add_variable(_var_qcd)

    _var_fake = Variable(r"Jet$\to\tau_h$ (fake)", is_independent=False, is_binned=False, units="Events")
    _var_fake.values = _cf_fake
    _var_fake.add_qualifier("SQRT(S)", 13, "TeV")
    _var_fake.add_qualifier("type", "background")
    table_cf.add_variable(_var_fake)

    submission.add_table(table_cf)


    ##############################
    # Signal efficiency map      #
    ##############################

    import re as _re
    import os as _os

    def _parse_eff_file(filepath):
        casc_dict, noncasc_dict = {}, {}
        with open(filepath) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                key, val = line.split(":")
                key = key.strip()
                val = float(val.strip().rstrip(","))
                m = _re.match(r"(Non)?Cascade-MA1-(\d+)_MA2-(\d+)", key)
                if not m:
                    continue
                m1, m2 = int(m.group(2)), int(m.group(3))
                if m.group(1) == "Non":
                    noncasc_dict[(m1, m2)] = val
                else:
                    casc_dict[(m1, m2)] = val
        entries = []
        for (m1, m2), eff in casc_dict.items():
            if (m1, m2) not in noncasc_dict:
                entries.append((m1, m2, eff))
        for (m1, m2), eff in noncasc_dict.items():
            entries.append((m1, m2, eff))
        entries.sort(key=lambda x: (x[0], x[1]))
        return entries

    _eff_base = "NPS25003_inputs/signal_efficiencies_Jun10"
    for _channel_dir in ["sigEff_emu", "sigEff_mutau", "sigEff_etau"]:
        _channel = _channel_dir[len("sigEff_"):]
        _eff_dir = _os.path.join(_eff_base, _channel_dir)
        for _eff_fname in sorted(_os.listdir(_eff_dir)):
            if not _eff_fname.endswith(".txt"):
                continue
            _eff_entries = _parse_eff_file(_os.path.join(_eff_dir, _eff_fname))
            _table_name = "Signal_efficiency_" + _os.path.splitext(_eff_fname)[0]
            _sr_m = _re.match(r'(SR(\d+))_((\d+)b)', _os.path.splitext(_eff_fname)[0])
            if _sr_m:
                _btag_desc = "exactly one b-tagged jet" if _sr_m.group(3) == "1b" else "at least two b-tagged jets"
                _sr_desc = "Signal Region " + _sr_m.group(2) + " with " + _btag_desc
            else:
                _sr_desc = None
            submission.add_table(makeEfficiencyTable(_table_name, _eff_entries, sr_description=_sr_desc, channel=_channel))

    for t in submission.tables:
        t.keywords["cmenergies"] = [13000]
    outdir = "NPS25003_output"
    print("Tables:", [t.name for t in submission.tables])
    
    submission.create_files(outdir, remove_old=True)


In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
!cat NPS25003_output/submission.yaml

In [ ]:
!ls NPS25003_output

In [ ]:
!ls submission.tar.gz